# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vxsnth/Machine_learning/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1. Unit of analysis + time window

One raw row represents one client-content page observation for one report date. For my lane, I will aggregate the daily data to one client-content page for the feature window.

I will use February 2026 as the feature window. These are the page-level signals that would be available when making the review-priority decision. I will use March 2026 as the future outcome window.

My lane is Refresh / Content Opportunity Scoring, so the final goal is to rank content pages for human review. I will keep the feature and outcome windows separate so future information does not enter the features.

In [13]:
import os
import getpass
import duckdb

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    return getpass.getpass("Enter your Hugging Face READ token: ")

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{get_hf_token()}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected successfully.")

Enter your Hugging Face READ token: ··········
Connected successfully.


In [14]:
# Check the February date range and raw row count

feb_window = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {FEB}
""").df()

feb_window

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,start_date,end_date
0,7355108,2026-02-01,2026-02-28


### 2. Fields: feature / label / context / excluded

**Features:** I will use five February page-level signals: GSC impressions, GSC clicks, GSC average position, GA4 pageviews, and GA4 engaged sessions. These are historical signals available before the March outcome window.

**Label:** I will use a future decline outcome as the label/proxy. A page is a positive outcome when its March search impressions decline relative to its February baseline. This is an observed performance outcome used for decision support, not proof that the page needs a refresh.

**Context:** `client_hash_id`, `content_hash_id`, `report_date`, and `month` identify the page and time period. The availability fields describe whether GSC or GA4 data is available.

**Excluded:** I will exclude future March performance from the February feature frame. I will also exclude `trend_direction` and `trend_pct` from the features because they are derived from performance trends and would leak outcome information if they overlap the label window.

## 3. Verify it with queries (grain, counts, missing values, windows)

**Verification 1 — Grain**

The raw fact table should have one row per client-content page per report date. I will compare the number of raw rows with the number of distinct client-content-date combinations.

In [15]:
grain_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS unique_pages,
    COUNT(
        DISTINCT client_hash_id
        || '|'
        || content_hash_id
        || '|'
        || CAST(report_date AS VARCHAR)
    ) AS unique_page_days
FROM {FEB}
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_pages,unique_page_days
0,7355108,321546,7355108


**Verification 2 — February window**

I am checking the number of February observations and the actual first and last report dates in the feature window.

In [16]:
window_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {FEB}
""").df()

window_check

,row_count,start_date,end_date
0,7355108,2026-02-01,2026-02-28


**Verification 3 — GSC availability**

I am checking how many February rows have GSC data explicitly available. I use `IS TRUE` so that only rows explicitly marked as available are counted as available.

In [17]:
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows
FROM {FEB}
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows
0,7355108,2621783


### Five features

I will use five features from the February 2026 feature window.

1. **GSC impressions** — knowable at the decision moment because search impressions were already observed during February.
2. **GSC clicks** — knowable at the decision moment because search clicks were already observed during February.
3. **GSC average position** — knowable at the decision moment because search-position observations were available during February.
4. **GA4 pageviews** — knowable at the decision moment because pageviews were already observed during February.
5. **GA4 engaged sessions** — knowable at the decision moment because engaged sessions were already observed during February.

All five features come from the February window, before the March outcome window.

In [18]:
features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions_feb,
    SUM(gsc_clicks) AS clicks_feb,
    AVG(gsc_avg_position) AS avg_position_feb,
    SUM(ga4_pageviews) AS pageviews_feb,
    SUM(ga4_engaged_sessions) AS engaged_sessions_feb
FROM {FEB}
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("Feature rows:", len(features))
print("Number of features:", 5)

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 153559
Number of features: 5


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,avg_position_feb,pageviews_feb,engaged_sessions_feb
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,12.946228,0.0,0.0
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.495085,6.0,0.0
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,10.490023,1.0,0.0
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,38.436254,9.0,1.0
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,9.710810,3.0,0.0


### The leakage trap

I will deliberately add March impressions to the February feature frame. This is a leaking feature because March impressions belong to the future outcome window and would not be available when making the February review-priority decision.

The purpose is to show how future information can make a quick score look unrealistically strong. After the experiment, I will remove the leaking feature and keep the honest feature set.

In [19]:
# Create the February baseline
feb_base = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions_feb
FROM {FEB}
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

# Create the March outcome
mar_outcome = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions_mar
FROM {MAR}
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

# Join February and March
outcomes = feb_base.merge(
    mar_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Define the future decline label
outcomes["decline_label"] = (
    outcomes["impressions_mar"] < outcomes["impressions_feb"]
).astype(int)

print("Outcome rows:", len(outcomes))
print("Decline rate:", round(outcomes["decline_label"].mean() * 100, 1), "%")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Outcome rows: 134238
Decline rate: 29.3 %


In [20]:
leaky_frame = features.merge(
    outcomes[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions_mar",
            "decline_label"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

leaky_frame.head()

,client_hash_id,content_hash_id,impressions_feb,clicks_feb,avg_position_feb,pageviews_feb,engaged_sessions_feb,impressions_mar,decline_label
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,12.946228,0.0,0.0,315.0,0
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.495085,6.0,0.0,14536.0,0
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,10.490023,1.0,0.0,387.0,1
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,38.436254,9.0,1.0,4697.0,0
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,9.710810,3.0,0.0,1004.0,0


In [24]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

feature_cols = [
    "impressions_feb",
    "clicks_feb",
    "avg_position_feb",
    "pageviews_feb",
    "engaged_sessions_feb"
]

leaky_cols = feature_cols + ["impressions_mar"]

X = leaky_frame[leaky_cols].fillna(0)
y = leaky_frame["decline_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leaky_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

leaky_model.fit(X_train, y_train)

leaky_pred = leaky_model.predict(X_test)

leaky_score = accuracy_score(y_test, leaky_pred)

print("Leaky accuracy:", round(leaky_score, 3))

Leaky accuracy: 0.776


### Removing the leakage

The leaky score is not trustworthy because `impressions_mar` comes from the future outcome window. It would not have been available when making the February decision.

I therefore remove `impressions_mar` and keep only the five February features. The honest feature set uses only information available before the March outcome.

In [22]:
honest_frame = leaky_frame.drop(
    columns=["impressions_mar"]
)

X = honest_frame[feature_cols].fillna(0)
y = honest_frame["decline_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

honest_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

honest_score = accuracy_score(y_test, honest_pred)

print("Leaky accuracy:", round(leaky_score, 3))
print("Honest accuracy:", round(honest_score, 3))
print("Leaking feature removed: impressions_mar")

Leaky accuracy: 0.776
Honest accuracy: 0.707
Leaking feature removed: impressions_mar


### 4. Data limits

This data can show observed relationships between earlier page performance and later search outcomes, but it cannot prove that refreshing a page will cause its performance to improve.

The warehouse has an unbalanced history, so different pages may have different amounts of historical data. GSC and GA4 availability can also vary across observations, so missing or unavailable data should not automatically be treated as zero.

The March decline label is only a performance proxy. A decline does not prove that a page needs a refresh, and a refresh is not guaranteed to improve performance.

I also keep the February feature window separate from the March outcome window. The final June 2026 month will not be used to develop the label logic because it should remain a sealed test period.

In [25]:
print("Feature window: February 2026")
print("Outcome window: March 2026")
print("Feature count:", len(feature_cols))
print("Future outcome fields excluded from features: impressions_mar")

Feature window: February 2026
Outcome window: March 2026
Feature count: 5
Future outcome fields excluded from features: impressions_mar


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.